Option Strategy Payoff Analyzer (Advanced)

This Project Calculates Payoff and profit/Loss for multiples option strategies.

Features:
Long Call / Long put
Bull Call Spread
Bear Put Spread
Long Straddle
Breakeven Points
Max Profit / Max Loss
Payoff Graph
Export Results to EXcel

In [ ]:
# Import Required Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# -----------------------------
# Strategy Menu Input
# -----------------------------
print("📌 Select an Option Strategy:")
print("1. Long Call")
print("2. Long Put")
print("3. Bull Call Spread")
print("4. Bear Put Spread")
print("5. Long Straddle")

strategy_choice = int(input("\nEnter choice (1-5): "))


# -----------------------------
# Spot Price Range Generator
# -----------------------------
def generate_spot_prices(base_strike, percent_range=30, step=10):
    lower = base_strike * (1 - percent_range / 100)
    upper = base_strike * (1 + percent_range / 100)
    return np.arange(lower, upper + step, step)


step = int(input("\nEnter step size for spot price (example 10, 20, 50): "))
percent_range = int(input("Enter % range around strike (example 30): "))


# -----------------------------
# Payoff Functions
# -----------------------------
def long_call_payoff(S, K):
    return np.maximum(S - K, 0)

def long_put_payoff(S, K):
    return np.maximum(K - S, 0)

def short_call_payoff(S, K):
    return -np.maximum(S - K, 0)

def short_put_payoff(S, K):
    return -np.maximum(K - S, 0)


# -----------------------------
# Strategy Logic
# -----------------------------
if strategy_choice == 1:
    print("\n✅ You selected: Long Call")

    strike_price = float(input("Enter strike price: "))
    premium = float(input("Enter premium paid: "))

    spot_prices = generate_spot_prices(strike_price, percent_range, step)

    payoff = long_call_payoff(spot_prices, strike_price)
    profit_loss = payoff - premium

    breakevens = [strike_price + premium]
    strategy_name = "Long Call"


elif strategy_choice == 2:
    print("\n✅ You selected: Long Put")

    strike_price = float(input("Enter strike price: "))
    premium = float(input("Enter premium paid: "))

    spot_prices = generate_spot_prices(strike_price, percent_range, step)

    payoff = long_put_payoff(spot_prices, strike_price)
    profit_loss = payoff - premium

    breakevens = [strike_price - premium]
    strategy_name = "Long Put"


elif strategy_choice == 3:
    print("\n✅ You selected: Bull Call Spread")

    K1 = float(input("Enter Strike Price (Buy Call - Lower Strike K1): "))
    K2 = float(input("Enter Strike Price (Sell Call - Higher Strike K2): "))

    premium_buy = float(input("Enter premium paid for buying call: "))
    premium_sell = float(input("Enter premium received for selling call: "))

    net_premium = premium_buy - premium_sell

    spot_prices = generate_spot_prices(K1, percent_range, step)

    payoff = long_call_payoff(spot_prices, K1) + short_call_payoff(spot_prices, K2)
    profit_loss = payoff - net_premium

    breakevens = [K1 + net_premium]
    strategy_name = "Bull Call Spread"


elif strategy_choice == 4:
    print("\n✅ You selected: Bear Put Spread")

    K1 = float(input("Enter Strike Price (Buy Put - Higher Strike K1): "))
    K2 = float(input("Enter Strike Price (Sell Put - Lower Strike K2): "))

    premium_buy = float(input("Enter premium paid for buying put: "))
    premium_sell = float(input("Enter premium received for selling put: "))

    net_premium = premium_buy - premium_sell

    spot_prices = generate_spot_prices(K1, percent_range, step)

    payoff = long_put_payoff(spot_prices, K1) + short_put_payoff(spot_prices, K2)
    profit_loss = payoff - net_premium

    breakevens = [K1 - net_premium]
    strategy_name = "Bear Put Spread"


elif strategy_choice == 5:
    print("\n✅ You selected: Long Straddle")

    strike_price = float(input("Enter strike price (same for call and put): "))
    call_premium = float(input("Enter premium paid for call: "))
    put_premium = float(input("Enter premium paid for put: "))

    total_premium = call_premium + put_premium

    spot_prices = generate_spot_prices(strike_price, percent_range, step)

    payoff = long_call_payoff(spot_prices, strike_price) + long_put_payoff(spot_prices, strike_price)
    profit_loss = payoff - total_premium

    breakevens = [strike_price - total_premium, strike_price + total_premium]
    strategy_name = "Long Straddle"


else:
    print("❌ Invalid choice! Please restart.")
    raise SystemExit


# -----------------------------
# Output Table with Status
# -----------------------------
df = pd.DataFrame({
    "Spot Price": spot_prices,
    "Payoff": payoff,
    "Profit/Loss": profit_loss
})

def status_label(x):
    if x > 0:
        return "Profit 🟢"
    elif x < 0:
        return "Loss 🔴"
    else:
        return "Breakeven ⚪"

df["Status"] = df["Profit/Loss"].apply(status_label)

print("\n📌 Payoff Table:")
print(df)


# -----------------------------
# Max Profit & Max Loss
# -----------------------------
max_profit = profit_loss.max()
max_loss = profit_loss.min()

print("\n📌 Strategy Summary")
print("===================================")
print(f"Strategy: {strategy_name}")
print(f"Max Profit: {max_profit:.2f}")
print(f"Max Loss: {max_loss:.2f}")
print(f"Breakeven Point(s): {', '.join([str(round(b,2)) for b in breakevens])}")
print("===================================")


# -----------------------------
# Plot Profit/Loss Graph
# -----------------------------
plt.figure(figsize=(10, 5))
plt.plot(spot_prices, profit_loss, label="Profit/Loss Curve")

# Zero profit line
plt.axhline(0, color="black", linestyle="--", linewidth=1)

# Breakeven lines
for be in breakevens:
    plt.axvline(be, linestyle="--", linewidth=1, label=f"Breakeven {be:.2f}")

plt.title(f"{strategy_name} Profit/Loss Graph")
plt.xlabel("Spot Price")
plt.ylabel("Profit / Loss")
plt.grid(True)
plt.legend()
plt.show()


# -----------------------------
# Export Table to Excel
# -----------------------------
file_name = f"{strategy_name.replace(' ', '_')}_Payoff.xlsx"
df.to_excel(file_name, index=False)
print(f"\n✅ Data exported successfully as: {file_name}")


# -----------------------------
# Export Table to CSV
# -----------------------------
csv_file = f"{strategy_name.replace(' ', '_')}_Payoff.csv"
df.to_csv(csv_file, index=False)
print(f"✅ Data exported successfully as: {csv_file}")
